In [1]:
import math


import torch
import torch.nn as nn
import torchvision
from torchvision import transforms

In [2]:
train_dataset = torchvision.datasets.ImageFolder(
    "datasets/butterflies",
    transform=transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ]
    ),
)

In [3]:
class PatchEmbed(nn.Module):
    """ """

    def __init__(
        self,
        img_size: int | tuple[int, int] = 224,
        patch_size: int | tuple[int, int] = 16,
        in_chans: int = 3,
        embed_dim: int = 768,
        bias: bool = True,
    ):
        super().__init__()

        if isinstance(img_size, int):
            img_size = (img_size, img_size)
        if isinstance(patch_size, int):
            patch_size = (patch_size, patch_size)

        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = tuple(
            [
                s // p
                for s, p in zip(
                    img_size,
                    patch_size,
                )
            ]
        )

        self.num_patches = self.grid_size[0] * self.grid_size[1]
        self.proj = nn.Conv2d(
            in_chans,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
            bias=bias,
        )

    def forward(self, x: torch.Tensor):
        # (B, C, H, W) <= x

        # B: batch
        # C: channel
        # H: height (original)
        # W: width (original)

        # D: embed_dim
        # h: height (projected)
        # w: width (projected)
        # L: seqlen = h*w

        # (B, D, h, w)
        x = self.proj(x)

        # (B, D, L)
        # (B, L, D)
        x = x.flatten(2).transpose(1, 2)

        return x


In [4]:
patch_embed = PatchEmbed()

img_tensor, label = train_dataset[0]
image_batch = img_tensor[None, :]
print(image_batch.shape)
image_projected = patch_embed(image_batch)
print(image_projected.shape)
print(patch_embed.grid_size)
print(patch_embed.num_patches)

torch.Size([1, 3, 224, 224])
torch.Size([1, 196, 768])
(14, 14)
196


In [5]:
class TimestepEmbedder(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        frequency_embedding_size: int = 256,
        use_bias=True,
    ):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(frequency_embedding_size, hidden_size, bias=use_bias),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size, bias=use_bias),
        )
        self.frequency_embedding_size = frequency_embedding_size

    @staticmethod
    def timestep_embedding(
        t: torch.Tensor,
        dim: int,
        max_period: int = 10000,
        freq_scale: int = 1,
    ):
        assert dim % 2 == 0

        half = dim // 2
        freqs = freq_scale * torch.exp(
            #####
            -math.log(max_period) * torch.arange(0, half, dtype=torch.float32) / half
        ).to(
            device=t.device,
        )

        #     t: (B,)
        # freqs: (half,)
        #  args: (B, half)
        args = t[:, None].float() * freqs[None, :]
        # embedding: (B, dim)
        embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        return embedding

    def forward(
        self,
        t: torch.Tensor,
        freq_scale: int = 1,
    ):
        #      t: (B,)
        # t_freq: (B, frequency_embedding_size)
        #  t_emb: (B, D)
        t_freq = self.timestep_embedding(
            t, self.frequency_embedding_size, freq_scale=freq_scale
        )
        t_emb = self.mlp(t_freq)
        return t_emb


In [6]:
timestep_embedder = TimestepEmbedder(768)
timestep_embedder(
    torch.tensor(
        [0, 100, 900],
    )
)

tensor([[ 0.1382,  0.0227,  0.0584,  ...,  0.0901, -0.1909,  0.0182],
        [ 0.1431,  0.0225,  0.1843,  ..., -0.0040,  0.0311, -0.0009],
        [-0.1108, -0.0683, -0.1565,  ..., -0.1234, -0.1559, -0.1001]],
       grad_fn=<AddmmBackward0>)

In [19]:
class LabelEmbedder(nn.Module):
    def __init__(
        self,
        num_classes: int,
        hidden_size: int,
        dropout_prob: float,
    ):
        super().__init__()
        use_cfg_embedding = 1 if (dropout_prob > 0.0) else 0
        self.embedding_table = nn.Embedding(
            num_classes + use_cfg_embedding,
            hidden_size,
        )
        self.num_classes = num_classes
        self.dropout_prob = dropout_prob

    def token_drop(
        self,
        labels: torch.Tensor,
        force_drop_mask: torch.Tensor = None,
    ):
        # force_drop_mask: 1D mask to labels
        if force_drop_mask is None:
            drop_mask = (
                torch.rand(labels.shape[0], device=labels.device) < self.dropout_prob
            )
        else:
            assert force_drop_mask.shape == labels.shape
            drop_mask = force_drop_mask.bool()

        labels = torch.where(drop_mask, self.num_classes, labels)
        return labels

    def forward(
        self,
        labels: torch.Tensor,
        train: bool,
        force_drop_mask: torch.Tensor = None,
    ):
        use_dropout = (
            #####
            self.dropout_prob > 0
        )
        if (train and use_dropout) or (force_drop_mask is not None):
            labels = self.token_drop(
                labels=labels,
                force_drop_mask=force_drop_mask,
            )
        embeddings = self.embedding_table(labels)
        return embeddings


In [25]:
label_embedder = LabelEmbedder(
    num_classes=20,
    hidden_size=768,
    dropout_prob=0.8,
)
label_embedder(
    torch.arange(0, 20),
    train=True,
    # force_drop_mask=torch.tensor([True] * 20),
)

tensor([[-0.4520, -0.3939,  0.9069,  ...,  1.0587, -0.1979, -0.0231],
        [-0.4520, -0.3939,  0.9069,  ...,  1.0587, -0.1979, -0.0231],
        [-0.4520, -0.3939,  0.9069,  ...,  1.0587, -0.1979, -0.0231],
        ...,
        [-0.4520, -0.3939,  0.9069,  ...,  1.0587, -0.1979, -0.0231],
        [-0.5667, -1.3652, -0.3729,  ...,  0.0073,  1.3450,  0.7999],
        [-0.4520, -0.3939,  0.9069,  ...,  1.0587, -0.1979, -0.0231]],
       grad_fn=<EmbeddingBackward0>)